# ADRE-CXR — Self-contained Kaggle Notebook

**No `src` upload needed.** Attach only one dataset:

> [Chest X-rays — Indiana University](https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university)

**Settings (right panel):**
- Accelerator: **GPU T4 x2** (or P100)
- Internet: **On**

This notebook:
1. Parses IU X-Ray reports → Findings / Impression
2. Creates dev (100) + test (400) splits
3. Extracts evidence packets with pretrained TorchXRayVision (**no training**)

Download `adre_cxr_processed.zip` when done → unzip to `adre-cxr/data/processed/` on your laptop.

In [ ]:
# --- 1. Install dependencies ---
!pip install -q torchxrayvision torch torchvision scikit-image opencv-python-headless tqdm pandas Pillow numpy

In [ ]:
# --- 2. Auto-detect dataset path ---
from pathlib import Path

CANDIDATES = [
    Path("/kaggle/input/chest-xrays-indiana-university"),
    Path("/kaggle/input/datasets/raddar/chest-xrays-indiana-university"),
    Path("/kaggle/input/raddar/chest-xrays-indiana-university"),
]

DATA_ROOT = None
for p in CANDIDATES:
    if (p / "indiana_reports.csv").exists():
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    for p in Path("/kaggle/input").rglob("indiana_reports.csv"):
        DATA_ROOT = p.parent
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Attach dataset: raddar/chest-xrays-indiana-university"
    )

WORK = Path("/kaggle/working/processed")
(WORK / "evidence").mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("Contents:", [x.name for x in sorted(DATA_ROOT.iterdir())])

In [ ]:
# --- 3. Self-contained helper code (no src import) ---
from __future__ import annotations

import json
import random
import re
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
import torchxrayvision as xrv
from PIL import Image

SECTION_PATTERNS = {
    "findings": r"(?i)(findings|finding)[:\s]+(.*?)(?=impression|comparison|indication|$)",
    "impression": r"(?i)(impression|imp)[:\s]+(.*?)(?=findings|comparison|indication|$)",
    "comparison": r"(?i)(comparison|comp)[:\s]+(.*?)(?=indication|findings|impression|$)",
    "indication": r"(?i)(indication|ind)[:\s]+(.*?)(?=comparison|findings|impression|$)",
}

def _extract_section(text, pattern):
    if not isinstance(text, str):
        return ""
    m = re.search(pattern, text, re.DOTALL)
    return m.group(2).strip() if m else ""

def parse_report_row(row):
    cols = {c.lower(): c for c in row.index}
    if "findings" in cols and "impression" in cols:
        findings = str(row[cols["findings"]]).strip()
        impression = str(row[cols["impression"]]).strip()
        comparison = str(row.get(cols.get("comparison", ""), "")).strip() if "comparison" in cols else ""
        indication = str(row.get(cols.get("indication", ""), "")).strip() if "indication" in cols else ""
    else:
        full = str(row.get("report", row.get("Report", "")))
        findings = _extract_section(full, SECTION_PATTERNS["findings"])
        impression = _extract_section(full, SECTION_PATTERNS["impression"])
        comparison = _extract_section(full, SECTION_PATTERNS["comparison"])
        indication = _extract_section(full, SECTION_PATTERNS["indication"])
    uid = str(row.get("uid", row.get("id", row.name)))
    return {"uid": uid, "comparison": comparison, "indication": indication,
            "findings": findings, "impression": impression}

def parse_reports_df(df, min_findings=10, min_impression=5):
    records = [parse_report_row(row) for _, row in df.iterrows()]
    return [r for r in records if len(r["findings"]) >= min_findings and len(r["impression"]) >= min_impression]

def make_splits(records, dev_size=100, test_size=400, seed=42):
    rng = random.Random(seed)
    shuffled = records.copy()
    rng.shuffle(shuffled)
    return shuffled[:dev_size], shuffled[dev_size:dev_size + test_size]

def save_splits(dev, test, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "dev_split.json").write_text(json.dumps(dev, indent=2))
    (out_dir / "test_split.json").write_text(json.dumps(test, indent=2))

@dataclass
class ImageView:
    view_id: str
    image: np.ndarray
    params: dict
    tool: str = "view_builder"

def load_grayscale(path):
    return np.array(Image.open(path).convert("L")).astype(np.float32)

def clahe_enhance(img, clip_limit=2.0):
    u8 = np.clip(img, 0, 255).astype(np.uint8)
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8)).apply(u8).astype(np.float32)

def simple_lung_bbox(img, margin=0.05):
    h, w = img.shape[:2]
    ys, xs = np.where(img > np.percentile(img, 20))
    if len(xs) == 0:
        return 0, 0, h, w
    y0, y1 = max(0, ys.min()), min(h, ys.max())
    x0, x1 = max(0, xs.min()), min(w, xs.max())
    dy, dx = int((y1 - y0) * margin), int((x1 - x0) * margin)
    return max(0, y0 - dy), min(h, y1 + dy), max(0, x0 - dx), min(w, x1 + dx)

def zone_crops(img, rows=3, cols=2):
    y0, y1, x0, x1 = simple_lung_bbox(img)
    lung = img[y0:y1, x0:x1]
    zh, zw = max(1, lung.shape[0] // rows), max(1, lung.shape[1] // cols)
    names = [("upper","left"),("upper","right"),("mid","left"),("mid","right"),("lower","left"),("lower","right")]
    views, idx = [], 0
    for r in range(rows):
        for c in range(cols):
            zy0, zy1 = r * zh, (r + 1) * zh if r < rows - 1 else lung.shape[0]
            zx0, zx1 = c * zw, (c + 1) * zw if c < cols - 1 else lung.shape[1]
            vrow, vcol = names[idx]
            views.append(ImageView(f"zone_{vrow}_{vcol}", lung[zy0:zy1, zx0:zx1],
                {"row": vrow, "col": vcol, "bbox": [y0, y1, x0, x1]}, "zone_crop"))
            idx += 1
    return views

def build_views(image_path, rows=3, cols=2):
    original = load_grayscale(image_path)
    views = [ImageView("original", original, {}, "load"),
             ImageView("clahe", clahe_enhance(original), {"clip_limit": 2.0}, "clahe")]
    views.extend(zone_crops(original, rows, cols))
    return views

def extract_features(img):
    mid = img.shape[1] // 2
    left, right = img[:, :mid], img[:, mid:]
    h_l, _ = np.histogram(left.flatten(), bins=32, range=(0, 255), density=True)
    h_r, _ = np.histogram(right.flatten(), bins=32, range=(0, 255), density=True)
    symmetry = max(0.0, 1.0 - np.abs(h_l - h_r).sum() / 2.0)
    thresh = np.percentile(img, 60)
    row_means = img.mean(axis=1)
    center = img.shape[0] // 2
    ctr = float(row_means[center:].mean() / (row_means[:center].mean() + 1e-6))
    return {"opacity_density": float(np.mean(img > thresh)), "symmetry_index": symmetry,
            "ctr_proxy": ctr, "mean_intensity": float(img.mean()), "std_intensity": float(img.std())}

def assess_quality(img):
    p5, p95 = np.percentile(img, [5, 95])
    dr = p95 - p5
    exp_label = "underexposed" if dr < 80 else ("overexposed" if dr > 220 else "adequate")
    edge = float(np.abs(np.diff(img, axis=0)).mean())
    insp = "adequate" if edge > 12 else ("suboptimal" if edge > 8 else "poor")
    h, w = img.shape
    upper = img[:h // 4, :]
    imb = abs(upper[:, :w//2].mean() - upper[:, w//2:].mean()) / (upper.mean() + 1e-6)
    rot = "minimal" if imb < 0.1 else ("mild" if imb < 0.25 else "moderate")
    return {"dynamic_range": float(dr), "exposure_label": exp_label,
            "rib_edge_density": edge, "inspiration_label": insp,
            "clavicle_imbalance": float(imb), "rotation_label": rot}

import torchvision

class NeuralInference:
    def __init__(self, weights="densenet121-res224-nih"):
        self.model = xrv.models.DenseNet(weights=weights)
        self.model.eval()
        if torch.cuda.is_available():
            self.model = self.model.cuda()
        self.pathologies = list(self.model.pathologies)
        self.transform = torchvision.transforms.Compose([
            xrv.datasets.XRayCenterCrop(),
            xrv.datasets.XRayResizer(224),
        ])

    def _prepare_tensor(self, img):
        """Normalize + resize to 224x224 as required by torchxrayvision."""
        if img.ndim == 3:
            img = img.mean(axis=-1)  # RGB -> grayscale
        img_norm = xrv.datasets.normalize(img, 255)  # maxval=255, NOT (0, 255)
        if img_norm.ndim == 2:
            img_norm = img_norm[None, ...]  # (1, H, W)
        img_ready = self.transform(img_norm)
        t = torch.from_numpy(img_ready).unsqueeze(0)  # (1, 1, 224, 224)
        if torch.cuda.is_available():
            t = t.cuda()
        return t

    def predict(self, img):
        t = self._prepare_tensor(img)
        with torch.no_grad():
            probs = torch.sigmoid(self.model(t))[0].cpu().numpy()
        return {p: float(probs[i]) for i, p in enumerate(self.pathologies)}

    def regional_burden(self, zone_probs):
        out = {}
        for zid, probs in zone_probs.items():
            out[zid] = {
                "max_pathology_prob": max(probs.values()) if probs else 0.0,
                "mean_pathology_prob": float(np.mean(list(probs.values()))) if probs else 0.0,
                "top_pathology": max(probs, key=probs.get) if probs else "none",
            }
        return out

def to_json_safe(obj):
    """Convert numpy scalars/arrays to native Python types for JSON."""
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_json_safe(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

@dataclass
class EvidencePacket:
    study_id: str
    image_path: str
    filename: str = ""
    patient_metadata: dict = field(default_factory=dict)
    global_pathology_probs: dict = field(default_factory=dict)
    global_abnormality_score: float = 0.0
    regional_burden: dict = field(default_factory=dict)
    image_features: dict = field(default_factory=dict)
    quality_indicators: dict = field(default_factory=dict)
    provenance: list = field(default_factory=list)
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def save(self, path):
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(to_json_safe(asdict(self)), indent=2))

def build_evidence_packet(study_id, image_path, filename, patient_metadata, neural):
    views = build_views(str(image_path))
    original = next(v for v in views if v.view_id == "original")
    packet = EvidencePacket(
        study_id=study_id, image_path=str(image_path), filename=filename,
        patient_metadata=patient_metadata,
        image_features=extract_features(original.image),
        quality_indicators=assess_quality(original.image),
        provenance=[{"view_id": v.view_id, "tool": v.tool, "params": v.params} for v in views],
    )
    packet.global_pathology_probs = neural.predict(original.image)
    packet.global_abnormality_score = max(packet.global_pathology_probs.values())
    zone_probs = {v.view_id: neural.predict(v.image) for v in views if v.view_id.startswith("zone_")}
    packet.regional_burden = neural.regional_burden(zone_probs)
    return packet

def find_image_path(data_root, filename):
    for sub in ["images/images_normalized", "images", ""]:
        p = data_root / sub / filename if sub else data_root / filename
        if p.exists():
            return p
    return None

print("Helpers loaded. CUDA:", torch.cuda.is_available())

In [ ]:
# --- 4. Load CSVs + inspect columns ---
reports = pd.read_csv(DATA_ROOT / "indiana_reports.csv")
projections = pd.read_csv(DATA_ROOT / "indiana_projections.csv")
print("Reports columns:", list(reports.columns))
print("Projections columns:", list(projections.columns))
print("Reports:", len(reports), "| Projections:", len(projections))
reports.head(2)

# --- 5. Parse reports + dev/test splits ---
parsed = parse_reports_df(reports)
dev, test = make_splits(parsed, dev_size=100, test_size=400, seed=42)
save_splits(dev, test, WORK)
print(f"Valid reports: {len(parsed)}")
print(f"Dev: {len(dev)} | Test: {len(test)}")

In [ ]:
# --- 6. Evidence extraction (GPU) ---
# Tips: set MAX_RECORDS = 5 for a quick test first, then None for all.
#       run SPLIT = "dev" first, then "test".

from tqdm.notebook import tqdm

SPLIT = "dev"        # "dev" or "test"
MAX_RECORDS = 5   # e.g. 5 for smoke test

split_data = json.loads((WORK / f"{SPLIT}_split.json").read_text())
if MAX_RECORDS:
    split_data = split_data[:MAX_RECORDS]

# CORRECTED: Changed "nih-rsna-14" to a valid torchxrayvision weight string
neural = NeuralInference(weights="densenet121-res224-nih")
skipped, saved = 0, 0

for rec in tqdm(split_data, desc=f"Evidence [{SPLIT}]"):
    uid = rec["uid"]
    proj = projections[projections["uid"] == int(uid)]
    if proj.empty:
        skipped += 1
        continue

    if "Frontal" in proj["projection"].values:
        row = proj[proj["projection"] == "Frontal"].iloc[0]
    else:
        row = proj.iloc[0]

    img_path = find_image_path(DATA_ROOT, row["filename"])
    if img_path is None:
        skipped += 1
        continue

    out_path = WORK / "evidence" / f"{uid}.json"
    if out_path.exists():
        saved += 1
        continue

    packet = build_evidence_packet(
        study_id=str(uid),
        image_path=img_path,
        filename=row["filename"],
        patient_metadata={"indication": rec.get("indication", "")},
        neural=neural,
    )
    packet.save(out_path)
    saved += 1

print(f"Saved: {saved} | Skipped: {skipped}")

In [ ]:
# --- 7. Preview one evidence packet ---
sample_files = sorted((WORK / "evidence").glob("*.json"))
if sample_files:
    sample = json.loads(sample_files[0].read_text())
    print("Study:", sample["study_id"])
    print("Abnormality score:", sample["global_abnormality_score"])
    top5 = sorted(sample["global_pathology_probs"].items(), key=lambda x: -x[1])[:5]
    print("Top pathologies:", top5)
    print("Quality:", sample["quality_indicators"])
else:
    print("No evidence files — check Step 6.")

In [ ]:
# --- 8. Zip outputs for download ---
import shutil
shutil.make_archive("/kaggle/working/adre_cxr_processed", "zip", WORK)
print("Download: adre_cxr_processed.zip")
print("Evidence files:", len(list((WORK / "evidence").glob("*.json"))))

## After running

1. **Save Version → Save & Run All**
2. Download **`adre_cxr_processed.zip`** from the Output tab
3. Unzip to `adre-cxr/data/processed/` on your laptop
4. Run LLM pipeline locally (Groq/Gemini — no GPU):

```bash
python scripts/run_pipeline.py \
  --study-id <uid> \
  --image "data/raw/.../images/images_normalized/<file>.png" \
  --evidence-json "data/processed/evidence/<uid>.json"
```

**GPU time:** ~100 dev studies ≈ 15–25 min on T4. Run dev first, then set `SPLIT = "test"` for the full test set.